# Final retrieval evaluation


## 1. Setup

In [1]:
from pathlib import Path
import gc
import importlib
import json
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "doc_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

embeddings_module = importlib.reload(importlib.import_module("doc_rag.5_embeddings"))
retrieval = importlib.reload(importlib.import_module("doc_rag.7_retrieval"))
indexing = importlib.reload(importlib.import_module("doc_rag.6_indexing"))
image_embeddings = importlib.reload(
    importlib.import_module("doc_rag.10_image_embeddings")
)

QUERIES_FILE = PROJECT_ROOT / "data" / "rag" / "evaluations" / "queries.json"
OUTPUT_DIR = PROJECT_ROOT / "data" / "rag" / "evaluations"
RESULTS_FILE = OUTPUT_DIR / "final_retrieval_results.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = PROJECT_ROOT / "data" / "rag" / "processed_documents"
EMBEDDING_DIR = (
    embeddings_module.DEFAULT_EMBEDDING_ROOT
    / embeddings_module._safe_model_name(embeddings_module.DEFAULT_MODEL_NAME)
)

# Locked production stack (from notebooks 3–5 / image bake-off)
TEXT_METHOD =  "hybrid_reranked_theme_reorder"
TEXT_MODEL_NAME = embeddings_module.DEFAULT_MODEL_NAME  # BAAI/bge-m3
RERANKER_NAME = retrieval.DEFAULT_RERANKER_MODEL
TOP_K = retrieval.DEFAULT_TOP_K
CANDIDATE_K = retrieval.DEFAULT_CANDIDATE_K

IMAGE_METHOD = "retrieve_rerank_fallback"
IMAGE_MODEL_NAME = retrieval.DEFAULT_IMAGE_CONTEXT_MODEL
IMAGE_CANDIDATE_K = 15
IMAGE_CONFIDENCE_THRESHOLD = 0.82 
IMAGE_CONTEXT_EMBED_ROOT = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "image_embedding_evaluation" / "embeddings"
)
IMAGE_STAGE_EMBED_ROOT = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "image_embedding_evaluation" / "stage_embeddings"
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("Queries:", QUERIES_FILE)
print("Results:", RESULTS_FILE)
print("Text method:", TEXT_METHOD, "-", TEXT_MODEL_NAME)
print(
    "Image method:", IMAGE_METHOD,
    "| candidate_k=", IMAGE_CANDIDATE_K,
    "| confidence=", IMAGE_CONFIDENCE_THRESHOLD,
)
print("Image context cache:", IMAGE_CONTEXT_EMBED_ROOT)
print("Image stage cache:", IMAGE_STAGE_EMBED_ROOT)

Device: cuda
Queries: C:\Users\User\Desktop\inmind\final project\services\gatherly_rag\data\rag\evaluations\queries.json
Results: C:\Users\User\Desktop\inmind\final project\services\gatherly_rag\data\rag\evaluations\final_retrieval_results.json
Text method: hybrid_reranked_theme_reorder - BAAI/bge-m3
Image method: retrieve_rerank_fallback | candidate_k= 15 | confidence= 0.82
Image context cache: C:\Users\User\Desktop\inmind\final project\services\gatherly_rag\data\rag\experiments\image_embedding_evaluation\embeddings
Image stage cache: C:\Users\User\Desktop\inmind\final project\services\gatherly_rag\data\rag\experiments\image_embedding_evaluation\stage_embeddings


## 2. Load queries from `queries.json`

In [2]:
raw_cases = json.loads(QUERIES_FILE.read_text(encoding="utf-8"))
if not isinstance(raw_cases, list) or not raw_cases:
    raise ValueError(f"No evaluation cases in {QUERIES_FILE}")

cases = []
for case in raw_cases:
    if not bool(case.get("answerable", True)):
        continue
    query_type = str(case.get("query_type", "text")).casefold()
    if query_type not in {"text", "text_image", "image"}:
        continue
    case = dict(case)
    case["query_type"] = query_type
    case["has_page_labels"] = bool(case.get("expected_pages"))
    case["has_image_labels"] = bool(case.get("expected_image_ids"))
    cases.append(case)

if pd.Series([c["id"] for c in cases]).duplicated().any():
    raise ValueError("Duplicate case ids in queries.json")

text_cases = [
    c for c in cases
    if c["query_type"] in {"text", "text_image"} and c.get("expected_document")
]
image_cases = [
    c for c in cases
    if c["query_type"] in {"image", "text_image"} and c.get("expected_image_ids")
]

print("Total answerable cases:", len(cases))
print("Text / text_image with expected_document:", len(text_cases))
print("Image / text_image with expected_image_ids:", len(image_cases))
display(pd.Series([c["query_type"] for c in cases]).value_counts().rename("cases"))
display(pd.Series([c.get("language", "?") for c in cases]).value_counts().rename("language"))

Total answerable cases: 175
Text / text_image with expected_document: 175
Image / text_image with expected_image_ids: 108


text_image    108
text           67
Name: cases, dtype: int64

en    130
ar     27
fr     18
Name: language, dtype: int64

## 3. Load text corpus (chunks + BGE embeddings + TF-IDF + reranker)

In [3]:
if not (EMBEDDING_DIR / "embeddings.npy").is_file():
    raise FileNotFoundError(
        f"Missing text embeddings at {EMBEDDING_DIR}. Rebuild BGE embeddings first."
    )

embedding_matrix = np.load(EMBEDDING_DIR / "embeddings.npy", allow_pickle=False)
saved_ids = json.loads((EMBEDDING_DIR / "chunks.json").read_text(encoding="utf-8"))[
    "chunk_ids"
]

chunk_rows = []
for path in PROCESSED_DIR.glob("*/chunks.json"):
    payload = json.loads(path.read_text(encoding="utf-8"))
    chunk_rows.extend(payload.get("chunks", []))
all_chunks = pd.DataFrame(chunk_rows)
chunks_by_id = all_chunks.set_index("chunk_id", drop=False)
missing_ids = [cid for cid in saved_ids if cid not in chunks_by_id.index]
if missing_ids:
    raise ValueError(f"Missing {len(missing_ids)} embedding-aligned chunks.")

chunks_df = chunks_by_id.loc[saved_ids].reset_index(drop=True)
if len(chunks_df) != embedding_matrix.shape[0]:
    raise ValueError("Chunk and embedding counts do not match.")

theme_registry = retrieval.build_theme_document_registry(chunks_df)
print("Theme registry size:", len(theme_registry))

model_kwargs = {"torch_dtype": torch.float16} if DEVICE == "cuda" else None
embedding_model = embeddings_module.load_embedding_model(
    TEXT_MODEL_NAME,
    device=DEVICE,
    local_files_only=True,
    model_kwargs=model_kwargs,
)
tfidf_index = retrieval.build_tfidf_index(chunks_df)
reranker = CrossEncoder(
    RERANKER_NAME,
    device=DEVICE,
    local_files_only=True,
    model_kwargs=model_kwargs,
)

print("Chunks:", len(chunks_df))
print("Embedding shape:", embedding_matrix.shape)
print("TF-IDF shape:", tfidf_index["matrix"].shape)
print("Reranker:", RERANKER_NAME)

Theme registry size: 12


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Chunks: 775
Embedding shape: (775, 1024)
TF-IDF shape: (775, 52320)
Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## 4. Final text evaluation (`hybrid_reranked`)

In [4]:
def acceptable_documents(case):
    return set(case.get("expected_documents") or [case["expected_document"]])


def relevant_pages_by_document(case):
    sources = case.get("relevant_sources") or []
    if sources:
        return {source["document"]: set(source.get("pages") or []) for source in sources}
    return {case["expected_document"]: set(case.get("expected_pages") or [])}


def text_row_is_relevant(case, row):
    file_name = row.get("file_name")
    if file_name not in acceptable_documents(case):
        return False

    # A supporting chunk may match an expected page, section, or verified answer keyword.
    expected_pages = relevant_pages_by_document(case).get(file_name, set())
    page = row.get("page_number")
    page_match = bool(expected_pages) and pd.notna(page) and int(page) in expected_pages

    expected_sections = {
        str(value).strip().casefold()
        for value in (case.get("expected_sections") or [])
        if str(value).strip()
    }
    section = str(row.get("section_title") or "").strip().casefold()
    section_match = bool(section) and any(
        expected in section or section in expected for expected in expected_sections
    )

    keywords = [
        str(value).strip().casefold()
        for value in (case.get("answer_keywords") or [])
        if str(value).strip()
    ]
    text = str(row.get("text") or "").casefold()
    keyword_match = any(keyword in text for keyword in keywords)

    if expected_pages or expected_sections or keywords:
        return bool(page_match or section_match or keyword_match)
    return True  # filename fallback only when no finer labels exist


def relevance_at_k(case, frame, k=TOP_K):
    values = [int(text_row_is_relevant(case, row)) for _, row in frame.head(k).iterrows()]
    return values + [0] * (k - len(values))


def ndcg_at_k(relevance):
    dcg = sum(value / np.log2(rank + 2) for rank, value in enumerate(relevance))
    ideal = sorted(relevance, reverse=True)
    idcg = sum(value / np.log2(rank + 2) for rank, value in enumerate(ideal))
    return dcg / idcg if idcg else 0.0


def text_hit_ranks(case, frame):
    expected_docs = acceptable_documents(case)
    relevant_pages = relevant_pages_by_document(case)
    document_rank = None
    page_rank = None
    for _, row in frame.iterrows():
        rank = int(row["rank"])
        file_name = row.get("file_name")
        if file_name not in expected_docs:
            continue
        if document_rank is None:
            document_rank = rank
        expected_pages = relevant_pages.get(file_name, set())
        if expected_pages and int(row.get("page_number")) in expected_pages:
            if page_rank is None:
                page_rank = rank
    relevant_rank = None
    for _, row in frame.iterrows():
        if text_row_is_relevant(case, row):
            relevant_rank = int(row["rank"])
            break
    return document_rank, page_rank, relevant_rank


text_case_rows = []
text_detail_rows = []

for case in text_cases:
    started = time.perf_counter()
    frame = retrieval.retrieve_hybrid_reranked(
        case["query"],
        embedding_model,
        embedding_matrix,
        chunks_df,
        tfidf_index,
        reranker,
        top_k=TOP_K,
        candidate_k=CANDIDATE_K,
        model_name=TEXT_MODEL_NAME,
    )
    frame = retrieval.apply_optional_theme_rerank(
        case["query"],
        frame,
        theme_registry,
        top_k=TOP_K,
        boost=retrieval.DEFAULT_THEME_RERANK_LAMBDA,
    )
    elapsed_ms = (time.perf_counter() - started) * 1000
    document_rank, page_rank, relevant_rank = text_hit_ranks(case, frame)
    relevance = relevance_at_k(case, frame)
    text_case_rows.append({
        "case_id": case["id"],
        "query": case["query"],
        "query_type": case["query_type"],
        "language": case.get("language"),
        "variant_of": case.get("variant_of"),
        "cross_language": bool(case.get("cross_language", False)),
        "expected_document": case["expected_document"],
        "expected_documents": sorted(acceptable_documents(case)),
        "has_page_labels": case["has_page_labels"],
        "modality": "text",
        "method": TEXT_METHOD,
        "document_rank": document_rank,
        "page_rank": page_rank,
        "relevant_rank": relevant_rank,
        "ndcg_at_5": ndcg_at_k(relevance),
        "latency_ms": elapsed_ms,
    })
    for _, row in frame.iterrows():
        text_detail_rows.append({
            "case_id": case["id"],
            "modality": "text",
            "method": TEXT_METHOD,
            "rank": int(row["rank"]),
            "score": float(row["score"]),
            "chunk_id": row.get("chunk_id"),
            "file_name": row.get("file_name"),
            "page_number": row.get("page_number"),
            "section_title": row.get("section_title"),
            "text": row.get("text"),
        })

text_case_results = pd.DataFrame(text_case_rows)
text_details = pd.DataFrame(text_detail_rows)
print("Text cases evaluated:", len(text_case_results))

Text cases evaluated: 175


In [5]:
def summarize_text(group):
    page_group = group[group["has_page_labels"]]
    return pd.Series({
        "cases": len(group),
        "Grounded Recall@1": group["relevant_rank"].le(1).fillna(False).mean(),
        "Grounded Recall@5": group["relevant_rank"].le(5).fillna(False).mean(),
        "Grounded MRR@5": (
            (1.0 / group["relevant_rank"]).where(group["relevant_rank"].le(5), 0.0).fillna(0.0).mean()
        ),
        "Grounded NDCG@5": group["ndcg_at_5"].mean(),
        "Document Recall@1": group["document_rank"].le(1).fillna(False).mean(),
        "Document Recall@5": group["document_rank"].le(5).fillna(False).mean(),
        "Page Recall@5": (
            page_group["page_rank"].le(5).fillna(False).mean()
            if len(page_group) else np.nan
        ),
        "failures": group["relevant_rank"].isna().sum() + group["relevant_rank"].gt(5).sum(),
        "mean_latency_ms": group["latency_ms"].mean(),
    })


text_overall = summarize_text(text_case_results).to_frame().T
text_by_document = (
    text_case_results.groupby("expected_document", sort=False)
    .apply(summarize_text, include_groups=False)
    .reset_index()
)
text_by_language = (
    text_case_results.groupby("language", sort=False)
    .apply(summarize_text, include_groups=False)
    .reset_index()
)
cross_language_overall = summarize_text(
    text_case_results[text_case_results["cross_language"]]
).to_frame().T
text_failures = text_case_results[
    text_case_results["relevant_rank"].isna()
    | text_case_results["relevant_rank"].gt(5)
].sort_values(["expected_document", "case_id"])

display(Markdown("### Text overall"))
display(text_overall)
display(Markdown("### Text by document"))
display(text_by_document)
display(Markdown("### Text by query language"))
display(text_by_language)
display(Markdown("### Cross-language subset"))
display(cross_language_overall)
display(Markdown("### Text failures (no grounded relevant result @5)"))
display(text_failures[["case_id", "query", "expected_document", "document_rank", "page_rank", "relevant_rank"]])

### Text overall

,cases,Grounded Recall@1,Grounded Recall@5,Grounded MRR@5,Grounded NDCG@5,Document Recall@1,Document Recall@5,Page Recall@5,failures,mean_latency_ms
0,175.0,0.748571,0.948571,0.83181,0.854865,0.948571,1.0,0.937143,9.0,236.352242


### Text by document

,expected_document,cases,Grounded Recall@1,Grounded Recall@5,Grounded MRR@5,Grounded NDCG@5,Document Recall@1,Document Recall@5,Page Recall@5,failures,mean_latency_ms
0,Organiser_un_evenement_deAaZ.pdf,18.0,0.666667,0.944444,0.777778,0.806916,1.000000,1.0,0.944444,1.0,290.325544
1,تقاليد الزفاف في الثقافات العربية.pdf,27.0,0.777778,0.962963,0.840741,0.854138,0.962963,1.0,0.925926,1.0,274.215578
2,Checklists.pdf,9.0,0.888889,1.000000,0.925926,0.919201,0.888889,1.0,0.888889,0.0,221.171533
3,Guideline_Sustainable_Event.pdf,13.0,0.769231,1.000000,0.884615,0.915507,0.923077,1.0,1.000000,0.0,216.009915
4,Boho_Wedding.pdf,16.0,0.687500,0.875000,0.781250,0.807242,1.000000,1.0,0.875000,2.0,226.205013
5,Romantic_Wedding.pdf,7.0,0.857143,0.857143,0.857143,0.857143,1.000000,1.0,0.857143,1.0,216.143414
6,Whimsical_Wedding.pdf,5.0,0.800000,1.000000,0.900000,0.926186,1.000000,1.0,1.000000,0.0,236.076080
7,Garden_Wedding.pdf,7.0,1.000000,1.000000,1.000000,0.988532,1.000000,1.0,1.000000,0.0,216.098414
8,Vintage_Wedding.pdf,9.0,0.777778,1.000000,0.888889,0.917984,1.000000,1.0,1.000000,0.0,224.413900
9,Rustic_Wedding.pdf,8.0,0.375000,1.000000,0.666667,0.752965,1.000000,1.0,1.000000,0.0,221.603062


### Text by query language

,language,cases,Grounded Recall@1,Grounded Recall@5,Grounded MRR@5,Grounded NDCG@5,Document Recall@1,Document Recall@5,Page Recall@5,failures,mean_latency_ms
0,fr,18.0,0.722222,0.944444,0.805556,0.822047,1.000000,1.0,0.944444,1.0,284.846611
1,ar,27.0,0.777778,1.000000,0.859259,0.883301,0.962963,1.0,0.962963,0.0,274.466096
2,en,130.0,0.746154,0.938462,0.829744,0.853503,0.938462,1.0,0.930769,8.0,221.721682


### Cross-language subset

,cases,Grounded Recall@1,Grounded Recall@5,Grounded MRR@5,Grounded NDCG@5,Document Recall@1,Document Recall@5,Page Recall@5,failures,mean_latency_ms
0,6.0,0.666667,0.833333,0.75,0.760873,1.0,1.0,0.833333,1.0,242.629267


### Text failures (no grounded relevant result @5)

,case_id,query,expected_document,document_rank,page_rank,relevant_rank
66,IMG_B06,How should the couple leave at the end of a bo...,Boho_Wedding.pdf,1,NaN,NaN
164,IMG_B12,How should boho reception tables be styled ove...,Boho_Wedding.pdf,1,NaN,NaN
137,IMG_EF03,What guest favors match an enchanted forest th...,Enchanted_Forest_Wedding.pdf,1,NaN,NaN
86,IMG_M01,How can a modern wedding lean into a strong pa...,Modern_Wedding.pdf,1,NaN,NaN
126,IMG_MW04,What black tableware and taper candles add edg...,Moody_Wedding.pdf,1,NaN,NaN
2,M2_03,Je suis assez désorganisé mais j’aimerais trav...,Organiser_un_evenement_deAaZ.pdf,1,NaN,NaN
68,IMG_RW02,What does a fairytale carriage exit look like ...,Romantic_Wedding.pdf,1,NaN,NaN
121,IMG_T06,What refreshing tropical cocktails suit a warm...,Tropical_Wedding.pdf,1,NaN,NaN
169,XL_EN_AW01,I'm planning a Lebanese wedding and want it to...,تقاليد الزفاف في الثقافات العربية.pdf,1,NaN,NaN


## 5. Final image evaluation (`retrieve_rerank_fallback`)

In [6]:
image_case_results = pd.DataFrame()
image_details = pd.DataFrame()
image_overall = pd.DataFrame()
image_failures = pd.DataFrame()
image_status = "skipped_no_image_cases"


def _coverage_at_k(retrieved_ids, expected_ids, k):
    expected = set(expected_ids)
    if not expected:
        return np.nan
    return sum(1 for image_id in retrieved_ids[:k] if image_id in expected) / len(expected)




if not image_cases:
    print("No image-labeled cases in queries.json yet. Skipping image eval.")
else:
    try:
        # Free text models before loading image E5 (GPU headroom)
        try:
            del embedding_model, reranker
        except NameError:
            pass
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        image_records = image_embeddings.load_image_records()
        image_pack = image_embeddings.get_or_create_image_embeddings(
            image_records,
            output_root=IMAGE_CONTEXT_EMBED_ROOT,
            context_model_name=IMAGE_MODEL_NAME,
            device=DEVICE,
            local_files_only=True,
            include_ocr=False,
            force=False,
        )
        stage_pack = image_embeddings.get_or_create_image_stage_embeddings(
            image_records,
            output_root=IMAGE_STAGE_EMBED_ROOT,
            context_model_name=IMAGE_MODEL_NAME,
            device=DEVICE,
            local_files_only=True,
            force=False,
        )
        context_embeddings = image_pack["context_embeddings"]
        stage1_embeddings = stage_pack["stage1_embeddings"]
        stage_meta = stage_pack["metadata"]["images"]
        records_with_stages = []
        for record, meta in zip(image_records, stage_meta):
            merged = dict(record)
            merged["stage1_source"] = meta["stage1_source"]
            merged["stage2_source"] = meta["stage2_source"]
            merged["stage1_text"] = meta["stage1_text"]
            merged["stage2_text"] = meta["stage2_text"]
            records_with_stages.append(merged)

        id_to_section = {
            str(record["image_id"]): str(record.get("section_title") or "").strip()
            for record in records_with_stages
        }

        image_model = embeddings_module.load_embedding_model(
            IMAGE_MODEL_NAME,
            device=DEVICE,
            local_files_only=True,
            model_kwargs=model_kwargs,
        )

        image_ids = {record["image_id"] for record in image_records}
        for case in image_cases:
            missing = [
                image_id
                for image_id in case["expected_image_ids"]
                if image_id not in image_ids
            ]
            if missing:
                raise ValueError(f"Case {case['id']} missing images: {missing}")

        image_case_rows = []
        image_detail_rows = []
        for case in image_cases:
            expected = set(case["expected_image_ids"])
            expected_sections = {
                str(s).strip()
                for s in (case.get("expected_sections") or [])
                if str(s).strip()
            }
            if not expected_sections:
                expected_sections = {
                    id_to_section[image_id]
                    for image_id in case["expected_image_ids"]
                    if image_id in id_to_section and id_to_section[image_id]
                }
            started = time.perf_counter()
            frame = retrieval.retrieve_images(
                case["query"],
                method=IMAGE_METHOD,
                model=image_model,
                model_name=IMAGE_MODEL_NAME,
                image_records=records_with_stages,
                context_embeddings=context_embeddings,
                stage1_embeddings=stage1_embeddings,
                top_k=TOP_K,
                candidate_k=IMAGE_CANDIDATE_K,
                confidence_threshold=IMAGE_CONFIDENCE_THRESHOLD,
            )
            elapsed_ms = (time.perf_counter() - started) * 1000
            retrieved_ids = frame["image_id"].tolist() if len(frame) else []
            retrieved_sections = (
                frame["section_title"].tolist()
                if len(frame) and "section_title" in frame.columns
                else [id_to_section.get(image_id, "") for image_id in retrieved_ids]
            )
            hit_rank = None
            for _, row in frame.iterrows():
                if row.get("image_id") in expected:
                    hit_rank = int(row["rank"])
                    break
            cov = _coverage_at_k(retrieved_ids, case["expected_image_ids"], TOP_K)
         
            image_case_rows.append({
                "case_id": case["id"],
                "query": case["query"],
                "query_type": case["query_type"],
                "language": case.get("language"),
                "expected_image_ids": list(case["expected_image_ids"]),
                "expected_sections": sorted(expected_sections),
                "modality": "image",
                "method": IMAGE_METHOD,
                "image_rank": hit_rank,
                "coverage_at_5": None if pd.isna(cov) else float(cov),
                "n_retrieved": len(retrieved_ids),
                "retrieved_image_ids": retrieved_ids,
                "latency_ms": elapsed_ms,
            })
            for _, row in frame.iterrows():
                image_detail_rows.append({
                    "case_id": case["id"],
                    "modality": "image",
                    "method": IMAGE_METHOD,
                    "rank": int(row["rank"]),
                    "score": float(row["score"]),
                    "image_id": row.get("image_id"),
                    "file_name": row.get("file_name"),
                    "section_title": row.get("section_title"),
                    "page_number": row.get("page_number"),
                })

        image_case_results = pd.DataFrame(image_case_rows)
        image_details = pd.DataFrame(image_detail_rows)
        image_overall = pd.DataFrame([{
            "cases": len(image_case_results),
            "Coverage@5": image_case_results["coverage_at_5"].mean(),
            "Image Recall@1": image_case_results["image_rank"].le(1).fillna(False).mean(),
            "Image Recall@5": image_case_results["image_rank"].le(5).fillna(False).mean(),
            "Image MRR@5": (
                (1.0 / image_case_results["image_rank"])
                .where(image_case_results["image_rank"].le(5), 0.0)
                .fillna(0.0)
                .mean()
            ),
            "mean_n_retrieved": image_case_results["n_retrieved"].mean(),
            "failures": (
                image_case_results["image_rank"].isna().sum()
                + image_case_results["image_rank"].gt(5).sum()
            ),
            "mean_latency_ms": image_case_results["latency_ms"].mean(),
        }])
        image_failures = image_case_results[
            image_case_results["image_rank"].isna()
            | image_case_results["image_rank"].gt(5)
        ]
        image_status = "completed"
        display(Markdown("### Image overall"))
        display(image_overall)

       
        display(Markdown("### Image failures (no gold id in top-5)"))
        display(image_failures[["case_id", "query", "image_rank"]])
        del image_model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    except Exception as error:
        image_status = f"skipped_error: {error}"
        print("Image eval skipped:", error)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Image overall

,cases,Coverage@5,Image Recall@1,Image Recall@5,Image MRR@5,mean_n_retrieved,failures,mean_latency_ms
0,108,0.914021,0.916667,0.916667,0.916667,1.990741,9,42.265719


### Image failures (no gold id in top-5)

,case_id,query,image_rank
6,IMG_RW01,How can I add romantic décor beyond just the a...,NaN
26,IMG_V07,What mixed vintage décor accents can dress a r...,NaN
64,IMG_MW03,How can wild greenery make a wedding feel dark...,NaN
68,IMG_MW07,What cocktail drinks match a dark gothic cockt...,NaN
71,IMG_CL02,What ceremony arch styles feel classic and ele...,NaN
76,IMG_EF03,What guest favors match an enchanted forest th...,NaN
94,IMG_RU05,What countryside bouquet styles fit a rustic w...,NaN
96,IMG_RU07,How can warm lighting make a barn wedding feel...,NaN
102,IMG_B11,How can boho seating help guests find their pl...,NaN


## 6. Combined overall (text + image)

Keeps the separate text/image tables above; this is the side-by-side final summary.

In [7]:
combined_rows = []
if len(text_overall):
    row = text_overall.iloc[0].to_dict()
    combined_rows.append({
        "modality": "text",
        "method": TEXT_METHOD,
        "cases": int(row.get("cases", 0)),
        "Recall@1": row.get("Grounded Recall@1"),
        "Recall@5": row.get("Grounded Recall@5"),
        "MRR@5": row.get("Grounded MRR@5"),
        "NDCG@5": row.get("Grounded NDCG@5"),
        "failures": row.get("failures"),
        "mean_latency_ms": row.get("mean_latency_ms"),
    })
if len(image_overall):
    row = image_overall.iloc[0].to_dict()
    combined_rows.append({
        "modality": "image",
        "method": IMAGE_METHOD,
        "cases": int(row.get("cases", 0)),
        "Coverage@5": row.get("Coverage@5"),
        "Recall@1": row.get("Image Recall@1"),
        "Recall@5": row.get("Image Recall@5"),
        "MRR@5": row.get("Image MRR@5"),
        "NDCG@5": None,
        "failures": row.get("failures"),
        "mean_latency_ms": row.get("mean_latency_ms"),
    })

combined_overall = pd.DataFrame(combined_rows)
if len(combined_overall):
    weight = combined_overall["cases"].astype(float)
    combined_overall = pd.concat(
        [
            combined_overall,
            pd.DataFrame([{
                "modality": "combined",
                "method": f"{TEXT_METHOD} + {IMAGE_METHOD}",
                "cases": int(weight.sum()),
                "Recall@1": (combined_overall["Recall@1"] * weight).sum() / weight.sum(),
                "Recall@5": (combined_overall["Recall@5"] * weight).sum() / weight.sum(),
                "MRR@5": (combined_overall["MRR@5"] * weight).sum() / weight.sum(),
                "NDCG@5": None,
                "failures": int(combined_overall["failures"].sum()),
                "mean_latency_ms": (
                    (combined_overall["mean_latency_ms"] * weight).sum() / weight.sum()
                ),
            }]),
        ],
        ignore_index=True,
    )

display(Markdown("### Combined overall"))
# display(combined_overall)
display(combined_overall.fillna("—"))
print("image_status:", image_status)


### Combined overall

,modality,method,cases,Recall@1,Recall@5,MRR@5,NDCG@5,failures,mean_latency_ms,Coverage@5
0,text,hybrid_reranked_theme_reorder,175,0.748571,0.948571,0.831810,0.854865,9.0,236.352242,—
1,image,retrieve_rerank_fallback,108,0.916667,0.916667,0.916667,—,9.0,42.265719,0.914021
2,combined,hybrid_reranked_theme_reorder + retrieve_reran...,283,0.812721,0.936396,0.864193,—,18.0,162.283887,—


image_status: completed


## 7. Save final results

In [8]:
payload = {
    "schema_version": 2,
    "evaluation_type": "final",
    "queries_file": str(QUERIES_FILE),
    "configuration": {
        "text_method": TEXT_METHOD,
        "text_embedding_model": TEXT_MODEL_NAME,
        "reranker_model": RERANKER_NAME,
        "top_k": TOP_K,
        "candidate_k": CANDIDATE_K,
        "image_method": IMAGE_METHOD,
        "image_embedding_model": IMAGE_MODEL_NAME,
        "image_candidate_k": IMAGE_CANDIDATE_K,
        "image_confidence_threshold": IMAGE_CONFIDENCE_THRESHOLD,
        "device": DEVICE,
    },
    "image_status": image_status,
    "text_overall": text_overall.where(pd.notna(text_overall), None).to_dict(
        orient="records"
    ),
    "text_by_document": text_by_document.where(pd.notna(text_by_document), None).to_dict(
        orient="records"
    ),
    "text_by_language": text_by_language.where(pd.notna(text_by_language), None).to_dict(
        orient="records"
    ),
    "cross_language_overall": cross_language_overall.where(
        pd.notna(cross_language_overall), None
    ).to_dict(orient="records"),
    "text_case_results": text_case_results.where(pd.notna(text_case_results), None).to_dict(
        orient="records"
    ),
    "text_details": text_details.where(pd.notna(text_details), None).to_dict(
        orient="records"
    ),
    "image_overall": image_overall.where(pd.notna(image_overall), None).to_dict(
        orient="records"
    ),
    "image_case_results": image_case_results.where(
        pd.notna(image_case_results), None
    ).to_dict(orient="records"),
    "image_details": image_details.where(pd.notna(image_details), None).to_dict(
        orient="records"
    ),
    "combined_overall": combined_overall.where(
        pd.notna(combined_overall), None
    ).to_dict(orient="records"),
}

RESULTS_FILE.write_text(
    json.dumps(payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved:", RESULTS_FILE)
print("Text cases:", len(text_case_results))
print("Image cases:", len(image_case_results))
print("Image status:", image_status)

for name in ("embedding_model", "reranker", "image_model"):
    if name in globals():
        del globals()[name]
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


Saved: C:\Users\User\Desktop\inmind\final project\services\gatherly_rag\data\rag\evaluations\final_retrieval_results.json
Text cases: 175
Image cases: 108
Image status: completed
